# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
## MODEL 4
### Author: Mariia Onokhina

---
### Experiment Results
* **Model:** HuggingFaceTB/SmolVLM-500M-Instruct, no fine-tuning
* **Inference:** Frozen model with multiple-choice log-likelihood scoring
* **Prompt:** `model2_imagecareful`
  * image token
  * instruction to answer a science multiple-choice question
  * added sentence: “Look carefully at the image when it is relevant.”
  * metadata: grade, subject, topic
  * hint if available
  * question
  * indexed choices
  * ends with `Answer:`
* **Scoring:** For each answer choice index `i`, append `" i"` to the prompt and compute answer-token log-likelihood only
* **Prediction:** `argmax` over index scores
* **Validation accuracy:** `0.5811`
* **Ensemble tried:** `model2_imagecareful + model2_category`, tied at `0.5811`, so **single prompt was chosen**
* **Result:** Best validation model so far; score 0.59154.

---
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [39]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [37]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ast
from transformers import AutoProcessor, AutoModelForVision2Seq

# For LoRa fine-tuning
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm   # For progress bar
from itertools import combinations

In [4]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
# Paths 
DATA_DIR = Path("../data")
IMAGE_ROOT = DATA_DIR / "images"

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 224

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [7]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

In [8]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

In [9]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [11]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [12]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

---
### Prompt Engineering

Play around with small variations to Model 2's prompt.

In [13]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

In [19]:
# Baseline build_prompt
def build_prompt_model2(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    prompt += f"Grade: {clean_text(row.get('grade', ''))}\n"
    prompt += f"Subject: {clean_text(row.get('subject', ''))}\n"
    prompt += f"Topic: {clean_text(row.get('topic', ''))}\n\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"Hint:\n{hint}\n\n"

    prompt += f"Question:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

Different prompt variants that are close to Model 2.

In [15]:
# Prompt variant 1
def build_prompt_model2_imagecareful(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Look carefully at the image when it is relevant.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    prompt += f"Grade: {clean_text(row.get('grade', ''))}\n"
    prompt += f"Subject: {clean_text(row.get('subject', ''))}\n"
    prompt += f"Topic: {clean_text(row.get('topic', ''))}\n\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"Hint:\n{hint}\n\n"

    prompt += f"Question:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

In [17]:
# Prompt variant 2
def build_prompt_model2_category(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    prompt += f"Grade: {clean_text(row.get('grade', ''))}\n"
    prompt += f"Subject: {clean_text(row.get('subject', ''))}\n"
    prompt += f"Topic: {clean_text(row.get('topic', ''))}\n"

    category = clean_text(row.get("category", ""))
    if category:
        prompt += f"Category: {category}\n"

    prompt += "\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"Hint:\n{hint}\n\n"

    prompt += f"Question:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

In [18]:
# Prompt variant 3
def build_prompt_model2_nohint(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    prompt += f"Grade: {clean_text(row.get('grade', ''))}\n"
    prompt += f"Subject: {clean_text(row.get('subject', ''))}\n"
    prompt += f"Topic: {clean_text(row.get('topic', ''))}\n\n"

    prompt += f"Question:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

In [20]:
PROMPT_BUILDERS = {
    "model2_exact": build_prompt_model2,
    "model2_imagecareful": build_prompt_model2_imagecareful,
    "model2_category": build_prompt_model2_category,
    "model2_nohint": build_prompt_model2_nohint,
}

---
### Model Loading

In [22]:
# The model is not related to Models 1, 2, and 3.
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
print("Model loaded.")

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded.


---
### Index-Only Scoring
Index-onluy scoring did better than hybrid index+answer scoring. 

This keeps the Model 2 idea, but uses safer answer-token masking.

In [23]:
@torch.no_grad()
def score_indices_for_row(row, prompt_builder, score_mode="sum"):
    image = Image.open(IMAGE_ROOT / row["image_path"]).convert("RGB")
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    answer_texts = [f" {i}" for i in range(num_choices)]
    full_texts = [prompt + ans for ans in answer_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in full_inputs.items()
    }

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)

    # Because logits[:, t] predicts token input_ids[:, t+1],
    # answer tokens begin around prompt_len - 1 in shifted space.
    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True

    # Ignore padding.
    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        elif score_mode == "mean":
            scores.append(vals.mean().item())
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

    return scores

In [24]:
def predict_row(row, prompt_builder, score_mode="sum"):
    scores = score_indices_for_row(row, prompt_builder, score_mode=score_mode)
    pred = int(np.argmax(scores))
    return pred, scores

---
### Test-Evaluate a Few Prompts

In [25]:
def evaluate_prompt(prompt_name, df=None, max_rows=None, score_mode="sum"):
    if df is None:
        df = val_df

    # If max_rows is not None, sample max_rows rows from the dataframe
    if max_rows is not None:
        eval_df = df.sample(max_rows, random_state=SEED).reset_index(drop=True)
    else:
        eval_df = df.reset_index(drop=True)

    prompt_builder = PROMPT_BUILDERS[prompt_name]

    preds = []
    labels = []
    all_scores = []

    # For each row in the dataframe, predict the answer and calculate the scores
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=prompt_name):
        pred, scores = predict_row(row, prompt_builder, score_mode=score_mode)
        preds.append(pred)
        labels.append(int(row["answer"]))
        all_scores.append(scores)

    preds = np.array(preds)
    labels = np.array(labels)
    acc = np.mean(preds == labels)

    print(f"{prompt_name} | {score_mode} | accuracy = {acc:.4f}")

    return {
        "prompt_name": prompt_name,
        "score_mode": score_mode,
        "accuracy": acc,
        "preds": preds,
        "labels": labels,
        "scores": all_scores,
        "eval_df": eval_df,
    }

In [26]:
# Get baseline result (Should be similar to Model 2)
baseline_result = evaluate_prompt(
    prompt_name="model2_exact",
    df=val_df,
    max_rows=None,
    score_mode="sum",
)

baseline_acc = baseline_result["accuracy"]
baseline_acc

model2_exact:   0%|          | 0/1048 [00:00<?, ?it/s]

model2_exact | sum | accuracy = 0.5677


np.float64(0.5677480916030534)

The baseline accuracy is 56.77%

---
### Search For Best Prompt On 200 Validation Rows

In [27]:
small_results = []

for prompt_name in PROMPT_BUILDERS.keys():
    result = evaluate_prompt(
        prompt_name=prompt_name,
        df=val_df,
        max_rows=200,
        score_mode="sum",
    )

    small_results.append({
        "prompt_name": prompt_name,
        "score_mode": "sum",
        "acc_200": result["accuracy"],
    })

small_results_df = pd.DataFrame(small_results).sort_values("acc_200", ascending=False)
small_results_df

model2_exact:   0%|          | 0/200 [00:00<?, ?it/s]

model2_exact | sum | accuracy = 0.5100


model2_imagecareful:   0%|          | 0/200 [00:00<?, ?it/s]

model2_imagecareful | sum | accuracy = 0.5050


model2_category:   0%|          | 0/200 [00:00<?, ?it/s]

model2_category | sum | accuracy = 0.5050


model2_nohint:   0%|          | 0/200 [00:00<?, ?it/s]

model2_nohint | sum | accuracy = 0.4950


,prompt_name,score_mode,acc_200
0,model2_exact,sum,0.510
1,model2_imagecareful,sum,0.505
2,model2_category,sum,0.505
3,model2_nohint,sum,0.495


There is currently no better prompt than Model 2's exact prompt.

Pick only variants that are close to or above Model 2 on the 200-row subset.

In [31]:
candidate_prompts = small_results_df[
    small_results_df["acc_200"] >= small_results_df["acc_200"].max() - 0.005
]["prompt_name"].tolist()

candidate_prompts

['model2_exact', 'model2_imagecareful', 'model2_category']

Run full validation for candidate prompts.

In [32]:
full_results = {}

for prompt_name in candidate_prompts:
    result = evaluate_prompt(
        prompt_name=prompt_name,
        df=val_df,
        max_rows=None,
        score_mode="sum",
    )

    full_results[prompt_name] = result

model2_exact:   0%|          | 0/1048 [00:00<?, ?it/s]

model2_exact | sum | accuracy = 0.5677


model2_imagecareful:   0%|          | 0/1048 [00:00<?, ?it/s]

model2_imagecareful | sum | accuracy = 0.5811


model2_category:   0%|          | 0/1048 [00:00<?, ?it/s]

model2_category | sum | accuracy = 0.5782


In [33]:
full_prompt_summary = pd.DataFrame([
    {
        "prompt_name": name,
        "accuracy": res["accuracy"],
    }
    for name, res in full_results.items()
]).sort_values("accuracy", ascending=False)

full_prompt_summary

,prompt_name,accuracy
1,model2_imagecareful,0.581107
2,model2_category,0.578244
0,model2_exact,0.567748


Model 2 prompt with an image-focused sentence scored 0.5811, which is much better than the baseline Model 2 prompt before training. Model 2 prompt with the category metadata field scored 0.5782, which is also better than the baseline.

---
### Error Analysis For Model 2

Visualize where model was off. Maybe it's a specific category that it keeps being inconsistent about.

In [34]:
def attach_predictions(result):
    df = result["eval_df"].copy()
    df["pred"] = result["preds"]
    df["label"] = result["labels"]
    df["correct"] = df["pred"] == df["label"]
    return df

baseline_pred_df = attach_predictions(baseline_result)
baseline_pred_df.head()

,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill,choices_list,pred,label,correct
0,val_00671,images/val/val_00671.png,Why might covering its eggs with its body incr...,"[""the leech's eggs will hatch"", ""the leech wil...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,"[the leech's eggs will hatch, the leech will n...",1,0,False
1,val_04111,images/val/val_04111.png,Why might fanning eggs increase the reproducti...,"[""the male will build a nest for females to la...",3,1,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,[the male will build a nest for females to lay...,1,1,True
2,val_02022,images/val/val_02022.png,"Based on clues in the text, how did fossil evi...","[""It helped scientists learn that beetle speci...",4,3,Read the text about beetles.\nThere are more s...,"Informational texts include many facts, exampl...",Think about these details from the text:\nFoss...,closed choice,grade5,language science,reading-comprehension,Informational texts: level 1,Read passages about animals,[It helped scientists learn that beetle specie...,2,3,False
3,val_01237,images/val/val_01237.png,What is the probability that a Cepaea snail pr...,"[""2/4"", ""4/4"", ""0/4"", ""3/4"", ""1/4""]",5,0,This passage describes the shell banding trait...,Offspring genotypes: homozygous or heterozygou...,NaN,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...,"[2/4, 4/4, 0/4, 3/4, 1/4]",0,0,True
4,val_03458,images/val/val_03458.png,What is the probability that a Cepaea snail pr...,"[""2/4"", ""0/4"", ""3/4"", ""1/4"", ""4/4""]",5,4,This passage describes the shell banding trait...,Offspring genotypes: homozygous or heterozygou...,NaN,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...,"[2/4, 0/4, 3/4, 1/4, 4/4]",0,4,False


In [35]:
for col in ["subject", "topic", "category", "skill", "grade"]:
    if col in baseline_pred_df.columns:
        print("\n" + "=" * 80)
        print(col)

        display(
            baseline_pred_df
            .groupby(col)
            .agg(
                n=("correct", "size"),
                acc=("correct", "mean"),
            )
            .query("n >= 5")
            .sort_values("acc")
            .head(20)
        )


subject


,n,acc
subject,,
natural science,777,0.553411
language science,28,0.571429
social science,243,0.613169



topic


,n,acc
topic,,
physics,213,0.399061
reading-comprehension,7,0.428571
geography,129,0.519380
chemistry,91,0.549451
earth-science,81,0.567901
biology,273,0.604396
writing-strategies,21,0.619048
world-history,11,0.636364
economics,64,0.656250



category


,n,acc
category,,
Classification,20,0.200000
Informational texts: level 1,5,0.200000
"Velocity, acceleration, and forces",51,0.235294
Magnets,63,0.269841
Genes to traits,46,0.326087
Cities,7,0.428571
Solutions,67,0.462687
Geography,60,0.500000
Adaptations,43,0.511628



skill


,n,acc
skill,,
"Identify mammals, birds, fish, reptiles, and amphibians",20,0.200000
Compare magnitudes of magnetic forces,70,0.228571
Use a letter-number grid,12,0.250000
Use Punnett squares to calculate probabilities of offspring types,22,0.272727
Compare strengths of magnetic forces,44,0.295455
Use Punnett squares to calculate ratios of offspring types,21,0.333333
Read passages about animals,6,0.333333
Compare ages of fossils in a rock sequence,8,0.375000
Compare concentrations of solutions,58,0.448276



grade


,n,acc
grade,,
grade3,106,0.452830
grade8,231,0.515152
grade4,172,0.575581
grade7,182,0.587912
grade5,119,0.588235
grade6,192,0.614583
grade2,36,0.722222
grade12,8,0.750000


The model has trouble solving physics and reading comprehension questions. It also has trouble identifying something from an image. Finally, surprisingly, it really struggles with lower grade questions. We can leave this for Model 5.

In [36]:
# Index bias check
print("True answer distribution:")
display(baseline_pred_df["label"].value_counts(normalize=True).sort_index())

print("Predicted answer distribution:")
display(baseline_pred_df["pred"].value_counts(normalize=True).sort_index())

True answer distribution:


label
0    0.331107
1    0.354962
2    0.235687
3    0.071565
4    0.006679
Name: proportion, dtype: float64

Predicted answer distribution:


pred
0    0.356870
1    0.286260
2    0.310115
3    0.046756
Name: proportion, dtype: float64

---
### Combining Several Scores

MAIN CHANGE OF MODEL 4.

Instead of trusting one prompt, we combine several prompts.

```
combined_score = prompt1_score + prompt2_score + prompt3_score
```

Then we choose the answer index with the highest combined score.

In [40]:
def normalize_scores_per_row(scores):
    scores = np.array(scores, dtype=np.float32)

    # Ignore scores that are too low
    valid = scores > -1e8
    normalized = scores.copy()

    # If there are no valid scores, return the original scores
    if valid.sum() <= 1:
        return normalized

    mean = normalized[valid].mean()
    std = normalized[valid].std()

    # If the standard deviation is too small, set it to 1.0
    if std < 1e-6:
        std = 1.0

    normalized[valid] = (normalized[valid] - mean) / std
    return normalized

In [41]:
# Combine scores from multiple prompts
def ensemble_from_results(result_list, normalize=True):
    labels = result_list[0]["labels"]
    eval_df = result_list[0]["eval_df"]

    ensemble_preds = []

    # For each row in the dataframe, combine the scores from multiple prompts
    for row_idx in range(len(eval_df)):
        num_choices = int(eval_df.iloc[row_idx]["num_choices"])
        combined_scores = np.zeros(num_choices, dtype=np.float32)

        # For each prompt, get the scores for the current row
        for result in result_list:
            scores = np.array(result["scores"][row_idx], dtype=np.float32)

            if normalize:
                scores = normalize_scores_per_row(scores)

            combined_scores += scores[:num_choices]

        # Choose the answer index with the highest combined score
        pred = int(np.argmax(combined_scores))
        ensemble_preds.append(pred)

    # Convert the list of predictions to an array
    ensemble_preds = np.array(ensemble_preds)
    acc = np.mean(ensemble_preds == labels)

    return acc, ensemble_preds

Run ensemble scores on 3 good prompt versions that we've just created.

In [42]:
ensemble_rows = []

# Try different combinations of prompts
for r in range(2, len(candidate_prompts) + 1):
    for combo in combinations(candidate_prompts, r):
        result_list = [full_results[name] for name in combo]

        acc_norm, preds_norm = ensemble_from_results(
            result_list,
            normalize=True
        )

        acc_raw, preds_raw = ensemble_from_results(
            result_list,
            normalize=False
        )

        ensemble_rows.append({
            "combo": combo,
            "normalize": True,
            "accuracy": acc_norm,
        })

        ensemble_rows.append({
            "combo": combo,
            "normalize": False,
            "accuracy": acc_raw,
        })

ensemble_df = pd.DataFrame(ensemble_rows).sort_values(
    "accuracy",
    ascending=False
)

ensemble_df

,combo,normalize,accuracy
4,"(model2_imagecareful, model2_category)",True,0.581107
5,"(model2_imagecareful, model2_category)",False,0.579198
1,"(model2_exact, model2_imagecareful)",False,0.577290
2,"(model2_exact, model2_category)",True,0.577290
6,"(model2_exact, model2_imagecareful, model2_cat...",True,0.577290
0,"(model2_exact, model2_imagecareful)",True,0.576336
7,"(model2_exact, model2_imagecareful, model2_cat...",False,0.574427
3,"(model2_exact, model2_category)",False,0.572519


Compare ensemble against best single prompt

In [43]:
full_prompt_summary = pd.DataFrame([
    {
        "prompt_name": name,
        "accuracy": result["accuracy"],
    }
    for name, result in full_results.items()
]).sort_values("accuracy", ascending=False)

display(full_prompt_summary)
display(ensemble_df)

,prompt_name,accuracy
1,model2_imagecareful,0.581107
2,model2_category,0.578244
0,model2_exact,0.567748


,combo,normalize,accuracy
4,"(model2_imagecareful, model2_category)",True,0.581107
5,"(model2_imagecareful, model2_category)",False,0.579198
1,"(model2_exact, model2_imagecareful)",False,0.577290
2,"(model2_exact, model2_category)",True,0.577290
6,"(model2_exact, model2_imagecareful, model2_cat...",True,0.577290
0,"(model2_exact, model2_imagecareful)",True,0.576336
7,"(model2_exact, model2_imagecareful, model2_cat...",False,0.574427
3,"(model2_exact, model2_category)",False,0.572519


Choose best model.

In [44]:
best_single_acc = full_prompt_summary.iloc[0]["accuracy"]
best_single_prompt = full_prompt_summary.iloc[0]["prompt_name"]

best_ensemble_acc = ensemble_df.iloc[0]["accuracy"]
best_ensemble_combo = list(ensemble_df.iloc[0]["combo"])
best_ensemble_normalize = bool(ensemble_df.iloc[0]["normalize"])

print("Best single:", best_single_prompt, best_single_acc)
print("Best ensemble:", best_ensemble_combo, best_ensemble_acc, "normalize =", best_ensemble_normalize)

if best_ensemble_acc > best_single_acc:
    final_mode = "ensemble"
    final_prompt_name = None
    final_ensemble_combo = best_ensemble_combo
    final_ensemble_normalize = best_ensemble_normalize
    best_val_acc = best_ensemble_acc
else:
    final_mode = "single"
    final_prompt_name = best_single_prompt
    final_ensemble_combo = None
    final_ensemble_normalize = None
    best_val_acc = best_single_acc

print()
print("FINAL MODEL 4")
print("final_mode:", final_mode)
print("final_prompt_name:", final_prompt_name)
print("final_ensemble_combo:", final_ensemble_combo)
print("final_ensemble_normalize:", final_ensemble_normalize)
print("best_val_acc:", best_val_acc)

Best single: model2_imagecareful 0.5811068702290076
Best ensemble: ['model2_imagecareful', 'model2_category'] 0.5811068702290076 normalize = True

FINAL MODEL 4
final_mode: single
final_prompt_name: model2_imagecareful
final_ensemble_combo: None
final_ensemble_normalize: None
best_val_acc: 0.5811068702290076


he ensemble does not beat the best single prompt. They both have the same accuracy score of 0.5811068702. I will choose the single prompt because it's simpler and faster.

Chosen prompt: `model2_imagecareful`

---
### PyTorch Dataset

No changes compared to the previous model.

In [45]:
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, image_root: Path, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.image_root = image_root
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        return Image.open(self.image_root / rel_path).convert("RGB")

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        item = {
            "id": row["id"],
            "image": img,
            "text": build_prompt_model2_imagecareful(row),
            "prompt": build_prompt_model2_imagecareful(row),
            "choices": row["choices_list"],
            "num_choices": int(row["num_choices"]),

        }
        
        if "answer" in row and pd.notna(row["answer"]):
            item["answer"] = int(row["answer"])
        else:
            item["answer"] = -1
        
        return item

# Create datasets
train_ds = ScienceQADataset(train_df, DATA_DIR, IMAGE_ROOT, is_train=True)
val_ds = ScienceQADataset(val_df, DATA_DIR, IMAGE_ROOT, is_train=False)
test_ds = ScienceQADataset(test_df, DATA_DIR, IMAGE_ROOT, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


---
### Evaluation

We will evaluate the best model so far:

In [46]:
final_mode = "single"
final_prompt_name = "model2_imagecareful"
best_val_acc = 0.5811068702290076

In [47]:
def predict_test_single(prompt_name):
    prompt_builder = PROMPT_BUILDERS[prompt_name]

    rows = []

    for _, row in tqdm(
        test_df.iterrows(),
        total=len(test_df),
        desc=f"test {prompt_name}"
    ):
        scores = score_indices_for_row(
            row,
            prompt_builder,
            score_mode="sum"
        )

        pred = int(np.argmax(scores))

        rows.append({
            "id": row["id"],
            "answer": pred,
        })

    return pd.DataFrame(rows)

In [48]:
submission = predict_test_single("model2_imagecareful")
display(submission.head())

test model2_imagecareful:   0%|          | 0/1008 [00:00<?, ?it/s]

,id,answer
0,test_01750,1
1,test_00128,0
2,test_02891,2
3,test_02425,2
4,test_00930,0


---
### Submission

In [49]:
# Make sure the submission is valid
def validate_submission(submission, test_df):
    assert list(submission.columns) == ["id", "answer"], "Columns must be id,answer"
    assert len(submission) == len(test_df), "Wrong number of rows"
    assert set(submission["id"]) == set(test_df["id"]), "Test ids do not match"

    submission["answer"] = submission["answer"].astype(int)

    # Merge the submission with the test dataframe to check if the answer is within the valid range
    merged = submission.merge(
        test_df[["id", "num_choices"]],
        on="id",
        how="left"
    )

    # Check if the answer is within the valid range
    bad = merged[
        (merged["answer"] < 0) |
        (merged["answer"] >= merged["num_choices"])
    ]

    print("Bad rows:", len(bad))

    if len(bad) > 0:
        display(bad.head(20))

    assert len(bad) == 0, "Invalid answer index found"

    print("Submission is valid.")


In [50]:
submission = submission[["id", "answer"]].copy()
submission["answer"] = submission["answer"].astype(int)

validate_submission(submission, test_df)

submission.to_csv("../submission.csv", index=False)

print("Saved submission.csv")
display(submission.head())
display(submission["answer"].value_counts(normalize=True).sort_index())

Bad rows: 0
Submission is valid.
Saved submission.csv


,id,answer
0,test_01750,1
1,test_00128,0
2,test_02891,2
3,test_02425,2
4,test_00930,0


answer
0    0.409722
1    0.249008
2    0.279762
3    0.058532
4    0.002976
Name: proportion, dtype: float64

This submission scored 0.59154.